In [1]:
import os
os.getcwd()

'c:\\Users\\param\\projects\\text-summarizer\\research'

In [2]:
os.chdir("../")
os.getcwd()

'c:\\Users\\param\\projects\\text-summarizer'

In [3]:
import os
import sys
import logging

from transformers import AutoTokenizer
from datasets import load_from_disk

from textSummarizer import logger
from textSummarizer.exception import CustomException
from textSummarizer.entity import DataTransformationConfig
from textSummarizer.config.configuration import ConfigurationManager

c:\Users\param\projects\text-summarizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [6]:
config_manager = ConfigurationManager()
data_transformation_config = config_manager.get_data_transformation_config()
print(data_transformation)


[2026-07-02 03:16:03,369: INFO: 26: common: yaml file (config\config.yaml) is loaded successfully]
[2026-07-02 03:16:03,375: INFO: 26: common: yaml file (params.yaml) is loaded successfully]
[2026-07-02 03:16:03,377: INFO: 43: common: Created directory at (artifacts)]
[2026-07-02 03:16:03,386: INFO: 43: common: Created directory at (artifacts/data_transformation)]
DataTransformationConfig(root_dir='artifacts/data_transformation', data_path='artifacts/data_ingestion/samsum_dataset', tokenizer_name='google/pegasus-cnn_dailymail')


In [ ]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig) -> DataTransformationConfig:
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(
            example_batch['dialogue'],
            max_length = 1028,
            truncation = True
        )
        target_encodings = self.tokenizer(
            example_batch['summary'],
            max_length = 128,
            truncation = True
        )
        return {
            'input_ids' : input_encodings['input_ids'],
            'attention_mask' : input_encodings['attention_mask'],
            'labels'  : target_encodings['input_ids']
        }
    
    def convert(self):
        try:
            
            dataset_samsum = load_from_disk(self.config.data_path)
            dataset_samsum_transformed = dataset_samsum.map(
                self.convert_examples_to_features,
                batched = True)
            dataset_samsum_transformed.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))
            logging.info("Dataset Transformation completed")

        except Exception as e:
            raise CustomException(e,sys)

In [9]:
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()

[2026-07-02 03:17:12,804: INFO: 1025: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-02 03:17:12,832: INFO: 1025: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-07-02 03:17:13,072: INFO: 1025: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-02 03:17:13,106: INFO: 1025: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-07-02 03:17:13,337: INFO: 1025: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 160858.02 examples/s]

[2026-07-02 03:17:17,434: INFO: 31: 1257517529: Dataset Transformation completed]
